# Drop Logger — Data Evaluation

Evaluate a downloaded drop logger CSV file.  This notebook covers:
- **Timing** — median / maximum sample interval, write-pause detection
- **Statistics** — min, max, mean, std for every measured channel
- **Time-series plots** — pressure, acceleration and rotation on a shared time axis
- **Sample rate** — instantaneous Hz over time to visualise write-flush pauses

**How to run:** open a terminal in `desktop-tools/` and launch `jupyter notebook`.  
Set `csv_path` in the *Load data* cell to evaluate a different file.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

In [ ]:
# -- set path to the CSV file to evaluate --
csv_path = Path('sample-data/droplogger-test.csv')
# -------------------------------------------

df_raw = pd.read_csv(csv_path)

# The unpack script writes a reference-pressure sentinel row at time = -0.001 s.
# Extract it if present, then work with data rows only.
ref_row = df_raw[df_raw['time(s)'] < 0]
if not ref_row.empty:
    ref_pressure_hpa = ref_row['Pressure Difference(hPa)'].iloc[0]
    print(f'Reference (absolute) pressure: {ref_pressure_hpa:.3f} hPa')
    df = df_raw[df_raw['time(s)'] >= 0].copy().reset_index(drop=True)
else:
    ref_pressure_hpa = None
    print('No reference pressure row found.')
    df = df_raw.copy().reset_index(drop=True)

t_start = df['time(s)'].iloc[0]
t_end   = df['time(s)'].iloc[-1]
print(f'Rows:     {len(df)}')
print(f'Time:     {t_start:.3f} s  to  {t_end:.3f} s  ({t_end - t_start:.3f} s duration)')
df.head()

## Timing Analysis

In [ ]:
dt = df['time(s)'].diff().dropna()

median_dt = dt.median()
mean_dt   = dt.mean()
max_dt    = dt.max()
min_dt    = dt.min()
std_dt    = dt.std()
median_hz = 1.0 / median_dt

print('=== Time Step Statistics ===')
print(f'  Median dt : {median_dt * 1000:7.3f} ms  ({median_hz:.0f} Hz)')
print(f'  Mean dt   : {mean_dt   * 1000:7.3f} ms')
print(f'  Std dt    : {std_dt    * 1000:7.3f} ms')
print(f'  Min dt    : {min_dt    * 1000:7.3f} ms')
print(f'  Max dt    : {max_dt    * 1000:7.3f} ms')

# Flag any time step > 5x median as a write-flush pause.
gap_threshold = median_dt * 5
gaps = dt[dt > gap_threshold]
print(f'\nWrite pauses (dt > 5x median = {gap_threshold * 1000:.1f} ms): {len(gaps)}')
for idx, gap in gaps.items():
    print(f'  t = {df["time(s)"].iloc[idx]:.3f} s   gap = {gap * 1000:.1f} ms')

## Channel Statistics

In [ ]:
# Compute total angular velocity as an extra summary channel
df['g_mag(deg/s)'] = np.sqrt(
    df['gX(deg/s)'] ** 2 + df['gY(deg/s)'] ** 2 + df['gZ(deg/s)'] ** 2
)

channels = {
    'Pressure Difference(hPa)' : 'Pressure diff    (hPa)',
    'a(ms^-2)'                 : 'Acceleration     (m/s^2)',
    'gX(deg/s)'                : 'Gyro X           (deg/s)',
    'gY(deg/s)'                : 'Gyro Y           (deg/s)',
    'gZ(deg/s)'                : 'Gyro Z           (deg/s)',
    'g_mag(deg/s)'             : 'Gyro magnitude   (deg/s)',
}

header = f"{'Channel':<28} {'Min':>10} {'Max':>10} {'Mean':>10} {'Std':>10}"
print(header)
print('-' * len(header))
for col, label in channels.items():
    s = df[col]
    print(f'{label:<28} {s.min():>10.2f} {s.max():>10.2f} {s.mean():>10.2f} {s.std():>10.2f}')

## Time-Series Plots

In [ ]:
t = df['time(s)']

fig, axes = plt.subplots(3, 1, sharex=True, figsize=(13, 10))
fig.suptitle(f'Drop Logger  |  {csv_path.name}', fontsize=13, fontweight='bold')

# Subplot 1 — pressure difference
axes[0].plot(t, df['Pressure Difference(hPa)'], color='steelblue', linewidth=0.8)
axes[0].set_ylabel('Pressure Diff (hPa)')
axes[0].set_title('Pressure Difference from Reference')

# Subplot 2 — acceleration magnitude
axes[1].plot(t, df['a(ms^-2)'], color='darkorange', linewidth=0.8)
axes[1].set_ylabel('Acceleration (m/s$^2$)')
axes[1].set_title('Acceleration Magnitude')

# Subplot 3 — gyroscope components + magnitude
axes[2].plot(t, df['gX(deg/s)'],    label='gX',  linewidth=0.8, color='#e41a1c')
axes[2].plot(t, df['gY(deg/s)'],    label='gY',  linewidth=0.8, color='#4daf4a')
axes[2].plot(t, df['gZ(deg/s)'],    label='gZ',  linewidth=0.8, color='#377eb8')
axes[2].plot(t, df['g_mag(deg/s)'], label='|g|', linewidth=1.0,
             color='black', linestyle='--', alpha=0.5)
axes[2].set_ylabel('Rotation (deg/s)')
axes[2].set_xlabel('Time (s)')
axes[2].set_title('Gyroscope Components')
axes[2].legend(loc='upper right', ncol=4)

plt.tight_layout()
plt.show()

## Instantaneous Sample Rate

Each dip corresponds to a write-flush pause (triggered every 500 rows).  
Sustained low rates or erratic patterns may indicate I2C contention.

In [ ]:
sample_rate = 1.0 / dt

fig, ax = plt.subplots(figsize=(13, 3))
ax.plot(df['time(s)'].iloc[1:], sample_rate, color='purple', linewidth=0.6, alpha=0.8)
ax.axhline(y=median_hz, color='red', linestyle='--', linewidth=1.2,
           label=f'Median: {median_hz:.0f} Hz')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Sample Rate (Hz)')
ax.set_title('Instantaneous Sample Rate')
ax.legend()
plt.tight_layout()
plt.show()

## Sensor Saturation Check

The ICM20649 gyro is configured at ±4000 deg/s.  Any sample at or near this limit has clipped data — rotational rates were higher than the sensor can measure.

In [ ]:
GYRO_LIMIT = 4000  # deg/s — hardware maximum at the configured range
SAT_MARGIN = 10    # flag samples within this many deg/s of the limit

for axis in ['gX(deg/s)', 'gY(deg/s)', 'gZ(deg/s)']:
    sat = df[df[axis].abs() >= (GYRO_LIMIT - SAT_MARGIN)]
    if sat.empty:
        print(f'{axis}: no saturation')
    else:
        print(f'{axis}: {len(sat)} saturated sample(s)  '
              f'(peak = {df[axis].abs().max():.0f} deg/s  '
              f'at t = {df.loc[df[axis].abs().idxmax(), "time(s)"]:.3f} s)')

## Suggestions for Further Analysis

### Drop dynamics
- **Pressure-derived altitude** — convert pressure difference to approximate altitude change: Δh ≈ Δp × 8.5 m/hPa (valid for small Δp near sea level).  This gives a relative altitude profile for the full drop.
- **Terminal velocity** — the pressure (and therefore altitude rate) stops changing once terminal velocity is reached.  The plateau in the pressure time series marks this point.
- **Tumble frequency (PSD)** — apply `scipy.signal.welch` to the gyro signals over the freefall window to characterise the dominant tumble frequency in Hz.
- **Impact severity** — the duration of the high-acceleration pulse at impact (above a threshold) combined with the peak value gives an indication of impact energy and surface hardness.

### Logger performance
- **Pressure noise floor** — compute the standard deviation of pressure samples during the post-landing phase (device stationary) as a real-world noise measurement in your operating conditions.
- **Multi-file comparison** — load several runs with `pd.concat` and overlay timing statistics to compare sample-rate consistency across units or battery states.
- **Write-pause impact** — check whether write pauses correlate with any artefacts in the sensor data (e.g. repeated values either side of a gap).